In [1]:
import pandas as pd

In [ ]:
# Outout the raw capital IQdataframe to a new Excel file to remove the Excel functions from each cell
# iron_mines.to_excel('../data/S&PGlobal iron ore mine economics/iron_mines_data.xlsx', index=False)

In [6]:
def capital_iq_property_data_processing(data_path: str) -> pd.DataFrame:

    # Load data
    capital_iq_data = pd.read_excel(data_path, sheet_name='Sheet1', index_col=False)
    capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'] = capital_iq_data['COMMODITY_PRODUCTION_TONNE_BY_PERIOD'] - (capital_iq_data['IRON_ORE_PRODUCTION_CONCENTRATE_TONNE'] + \
                                                                    capital_iq_data['IRON_ORE_PRODUCTION_LUMP_TONNE'] + capital_iq_data['IRON_ORE_PRODUCTION_FINES_TONNE'])
    # check if the new column has negative values, if so, change the negative values to 0
    capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'] = capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'].apply(lambda x: max(x, 0))

    # Put the new column after the column 'IRON_ORE_PRODUCTION_LUMP_TONNE'
    capital_iq_data.insert(capital_iq_data.columns.get_loc('IRON_ORE_PRODUCTION_LUMP_TONNE') + 1, 'IRON_ORE_PRODUCTION_PELLET_TONNE', capital_iq_data.pop('IRON_ORE_PRODUCTION_PELLET_TONNE'))

    # Filter the dataframe based on the column 'ACTV_STATUS' and change the status name from 'Active' to 'operating', 'Care And Maintenance' to 'mothballed', 'Rehabilitation' to 'mothballed', 'Temporarily On Hold' to 'proposed', 'On Hold Awaiting Higher Prices' to 'proposed', 'On Hold Awaiting Financing' to 'proposed',  'Under Litigation' to 'proposed'
    capital_iq_data = capital_iq_data[capital_iq_data['ACTV_STATUS'].isin(['Active', 'Care And Maintenance', 'Rehabilitation', 'Temporarily On Hold', 'On Hold Awaiting Higher Prices', 'On Hold Awaiting Financing', 'Under Litigation'])]
    capital_iq_data['ACTV_STATUS'] = capital_iq_data['ACTV_STATUS'].replace({'Active': 'operating', 'Care And Maintenance': 'mothballed', 'Rehabilitation': 'mothballed', 'Temporarily On Hold': 'proposed', 'On Hold Awaiting Higher Prices': 'proposed', 'On Hold Awaiting Financing': 'proposed',  'Under Litigation': 'proposed'})

    # ── Pivot product-level columns ───────────────────────────────────────────
    # Each product group maps its original column names to shared value_name labels.
    # product_name is the 3rd token (index 2) of the FE-content column name.
    PIVOT_GROUPS = {
        'CONCENTRATE': {
            'IRON_ORE_CONCENTRATE_FE_CONTENT_PCT':                                                       'fe_content_pct',
            'IRON_ORE_PRODUCTION_CONCENTRATE_TONNE':                                                     'production_tonne',
            'TOTAL_CASH_COST_DMT_CONCENTRATE':                                                           'total_cash_cost_dmt',
            'TOTAL_CASH_COST_CFR_DMT_CONCENTRATE':                                                       'total_cash_cost_cfr_dmt',
            'CONCENTRATE_BULKMETALS_SCOPE_1_2_TRANSPORTATION_EMISSIONS_PAID_METAL_PRODUCTION_EMISSIONS': 'scope_1_2_emissions',
        },
        'Fines': {
            'IRON_ORE_Fines_FE_CONTENT_PCT':                                                             'fe_content_pct',
            'IRON_ORE_PRODUCTION_FINES_TONNE':                                                           'production_tonne',
            'TOTAL_CASH_COST_DMT_FINES':                                                                 'total_cash_cost_dmt',
            'TOTAL_CASH_COST_CFR_DMT_FINES':                                                             'total_cash_cost_cfr_dmt',
            'FINES_BULKMETALS_SCOPE_1_2_TRANSPORTATION_EMISSIONS_PAID_METAL_PRODUCTION_EMISSIONS':       'scope_1_2_emissions',
        },
        'Lump': {
            'IRON_ORE_Lump_FE_CONTENT_PCT':                                                              'fe_content_pct',
            'IRON_ORE_PRODUCTION_LUMP_TONNE':                                                            'production_tonne',
            'TOTAL_CASH_COST_DMT_LUMP':                                                                  'total_cash_cost_dmt',
            'TOTAL_CASH_COST_CFR_DMT_LUMP':                                                              'total_cash_cost_cfr_dmt',
            'LUMP_BULKMETALS_SCOPE_1_2_TRANSPORTATION_EMISSIONS_PAID_METAL_PRODUCTION_EMISSIONS':        'scope_1_2_emissions',
        },
        'Pellet': {
            'IRON_ORE_Pellet_FE_CONTENT_PCT':                                                            'fe_content_pct',
            'IRON_ORE_PRODUCTION_PELLET_TONNE':                                                          'production_tonne',
            'TOTAL_CASH_COST_DMT_PELLET':                                                                'total_cash_cost_dmt',
            'TOTAL_CASH_COST_CFR_DMT_PELLET':                                                            'total_cash_cost_cfr_dmt',
            'PELLET_BULKMETALS_SCOPE_1_2_TRANSPORTATION_EMISSIONS_PAID_METAL_PRODUCTION_EMISSIONS':      'scope_1_2_emissions',
        },
    }

    all_pivot_cols = {c for group in PIVOT_GROUPS.values() for c in group}
    id_cols = [c for c in capital_iq_data.columns if c not in all_pivot_cols]

    product_frames = []
    for product_name, col_map in PIVOT_GROUPS.items():
        valid_map = {c: v for c, v in col_map.items() if c in capital_iq_data.columns}
        sub = capital_iq_data[id_cols + list(valid_map.keys())].copy()
        sub = sub.rename(columns=valid_map)
        sub['product_name'] = product_name
        product_frames.append(sub)

    melted = pd.concat(product_frames, ignore_index=True)

    # Keep only rows where FE content is present and non-zero
    melted = melted[
        melted['fe_content_pct'].notna() & (melted['fe_content_pct'] != 0)
    ].reset_index(drop=True)

    return melted

In [11]:
data_path = '../data/SPGlobalCaptialQ/'
df = capital_iq_property_data_processing(data_path + 'iron_ore_property.xlsx')
# df.to_excel(data_path + 'melted_capital_iq_data.xlsx', index=False)

In [25]:
capital_iq_data = pd.read_excel(data_path + 'iron_ore_property.xlsx', sheet_name='Sheet1', index_col=False)
capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'] = capital_iq_data['COMMODITY_PRODUCTION_TONNE_BY_PERIOD'] - (capital_iq_data['IRON_ORE_PRODUCTION_CONCENTRATE_TONNE'] + \
                                                                capital_iq_data['IRON_ORE_PRODUCTION_LUMP_TONNE'] + capital_iq_data['IRON_ORE_PRODUCTION_FINES_TONNE'])
# check if the new column has negative values, if so, change the negative values to 0
capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'] = capital_iq_data['IRON_ORE_PRODUCTION_PELLET_TONNE'].apply(lambda x: max(x, 0))

# Put the new column after the column 'IRON_ORE_PRODUCTION_LUMP_TONNE'
capital_iq_data.insert(capital_iq_data.columns.get_loc('IRON_ORE_PRODUCTION_LUMP_TONNE') + 1, 'IRON_ORE_PRODUCTION_PELLET_TONNE', capital_iq_data.pop('IRON_ORE_PRODUCTION_PELLET_TONNE'))

# Filter the dataframe based on the column 'ACTV_STATUS' and change the status name from 'Active' to 'operating', 'Care And Maintenance' to 'mothballed', 'Rehabilitation' to 'mothballed', 'Temporarily On Hold' to 'proposed', 'On Hold Awaiting Higher Prices' to 'proposed', 'On Hold Awaiting Financing' to 'proposed',  'Under Litigation' to 'proposed'
capital_iq_data = capital_iq_data[capital_iq_data['ACTV_STATUS'].isin(['Active', 'Care And Maintenance', 'Rehabilitation', 'Temporarily On Hold', 'On Hold Awaiting Higher Prices', 'On Hold Awaiting Financing', 'Under Litigation'])]
capital_iq_data['ACTV_STATUS'] = capital_iq_data['ACTV_STATUS'].replace({'Active': 'operating', 'Care And Maintenance': 'mothballed', 'Rehabilitation': 'mothballed', 'Temporarily On Hold': 'proposed', 'On Hold Awaiting Higher Prices': 'proposed', 'On Hold Awaiting Financing': 'proposed',  'Under Litigation': 'proposed'})


# count the number of rows in the column 'prop_name' where the total Fe content of all four product types is equal to 0
capital_iq_data['total_cash_cost_dmt_sum'] = capital_iq_data[['TOTAL_CASH_COST_DMT_CONCENTRATE', 'TOTAL_CASH_COST_DMT_FINES', 'TOTAL_CASH_COST_DMT_LUMP', 'TOTAL_CASH_COST_DMT_PELLET']].sum(axis=1)
zero_cash_cost_count = capital_iq_data[capital_iq_data['total_cash_cost_dmt_sum'] == 0]['PROP_NAME'].nunique()
print(f'Number of unique properties with zero total cash cost dmt: {zero_cash_cost_count}')

Number of unique properties with zero total cash cost dmt: 716


In [26]:
df2 = pd.read_excel(data_path + 'melted_capital_iq_data.xlsx', sheet_name='Sheet1', index_col=False)
# count the number of rows in the column 'prop_name'
unique_properties_count = df2['PROP_NAME'].nunique()
print(f'Number of unique properties: {unique_properties_count}')


Number of unique properties: 233


In [46]:
# Get location data from capital IQ (iron_ore_property.xlsx) and merge it with the mine asset matches data (mine_asset_matches_distance.xlsx) to get the lat and long of each mine asset match, save the merged data to a new Excel file

def get_location_from_property_data(data1, data2):
    iron_ore_property = pd.read_excel(data1, sheet_name='Sheet1', index_col=False)
    mine_asset_matches = pd.read_excel(data2, sheet_name='Sheet2', index_col=False)

    # Left join iron_ore_propery to mine_asset_matches on the column 'PROP_NAME' on iron_ore_propery and Mine Short Name (List 2) on mine_asset_matches, get the lat and long from iron_ore_property
    merged_df = pd.merge(mine_asset_matches, iron_ore_property[['PROP_NAME', 'LATITUDE', 'LONGITUDE', 'STATE_PROVINCE', 'COUNTRY_NAME']], how='left', left_on='Mine Short Name (List 2)', right_on='PROP_NAME')
    merged_df.drop(columns=['PROP_NAME'], inplace=True)
    # save to the sheet2 of mine_asset_matches_final.xlsx
    merged_df.to_excel('../data/SPGlobalCaptialQ/S&PGlobal iron ore mine economics/mine_asset_matches_distance.xlsx', sheet_name='Sheet2', index=False)
    return merged_df


data1 = '../data/SPGlobalCaptialQ/iron_ore_property.xlsx'
data2 = '../data/SPGlobalCaptialQ/S&PGlobal iron ore mine economics/mine_asset_matches_distance.xlsx'
df = get_location_from_property_data(data1, data2)




In [ ]:
# left join melted_capital_iq_data to  mine_distance_matches (mines with cost data) on prop_name, and add only product_name to the mine_distance_matches dataframe, then join based on both prop_name and product name, save the merged data to a new Excel file  
def merge_product_name_emisions_to_mine_distance_matches(data1, data2):
    melted_capital_iq_data = pd.read_excel(data1, sheet_name='Sheet1', index_col=False)
    mine_distance_matches = pd.read_excel(data2, sheet_name='Distance Match Results', index_col=False)

    # Left join melted_capital_iq_data to mine_distance_matches on prop_name, get the product_name and emissions from melted_capital_iq_data
    merged_df = pd.merge(mine_distance_matches, melted_capital_iq_data[['PROP_NAME', 'product_name', 'scope_1_2_emissions']], how='left', left_on=['capital_iq_mines'], right_on=['PROP_NAME'])
    merged_df.drop(columns=['PROP_NAME'], inplace=True)

    return merged_df

# join the merged_df to SPGlobal_MineEconomics_22-Apr-2026.xlsx on the column capital_iq_mines on right and Property Name on left, and get the global_tracker_mines, global_tracker_Operator, capital_iq_State/Province,product_name, emissions from merged_df, save the final merged data to a new Excel file
def merge_with_mine_econ(merged_df,data3):
    mine_econ = pd.read_excel(data3, sheet_name='Data', index_col=False, skiprows=12)
    final_merged_df = pd.merge(mine_econ, merged_df[['capital_iq_mines', 'product_name', 'scope_1_2_emissions', 'capital_iq_State/Province', 'global_tracker_mines', 'global_tracker_Operator']], how='left', left_on='Property', right_on='capital_iq_mines')
    final_merged_df.drop(columns=['capital_iq_mines'], inplace=True)
    # merge the correct_mine column with global_tracker_mines and correct_operator with global_tracker_Operator, if the global_tracker_mines column is not null, then use the value in global_tracker_mines column to fill the correct_mine column, if the global_tracker_Operator column is not null, then use the value in global_tracker_Operator column to fill the correct_operator column
    
    final_merged_df['Correct_mine'] = final_merged_df.apply(lambda x: x['global_tracker_mines'] if pd.notnull(x['global_tracker_mines']) else x['Correct_mine'], axis=1)
    final_merged_df['Correct_operator'] = final_merged_df.apply(lambda x: x['global_tracker_Operator'] if pd.notnull(x['global_tracker_Operator']) else x['Correct_operator'], axis=1)
    final_merged_df.drop(columns=['global_tracker_mines', 'global_tracker_Operator'], inplace=True)

    return final_merged_df

data1 = '../data/SPGlobalCaptialQ/melted_capital_iq_data.xlsx'
data2 = '../data/SPGlobalCaptialQ/mine_distance_matches (mines with cost data).xlsx'
data3 = '../data/SPGlobalCaptialQ/S&PGlobal iron ore mine economics/SPGlobal_MineEconomics_22-Apr-2026.xlsx'
df = merge_product_name_emisions_to_mine_distance_matches(data1, data2)

final_df = merge_with_mine_econ(df, data3)

final_df.to_excel('../data/SPGlobalCaptialQ/S&PGlobal iron ore mine economics/final_merged_data.xlsx', index=False)